# CML3111 Machine Learning: Engineering ML project, REHEARSAL (unmarked)
**This is a rehearsal. Nothing here is marked or submitted.** It has the same structure, the same task wording and the same kinds of check as the real project sitting, but a different practice data file (variant P), so your numbers will not be the numbers you get on the day. Released Friday 27 November 2026 (after Class 4; proposed, lecturer to confirm). The real sitting is Friday 4 December 2026, 11:00 to 14:00 Kampala time (3 hours; proposed, lecturer to confirm).

**How to use it.** Open it in Colab (File, then Save a copy in Drive). Work through it as if it were the real sitting: set a timer for 3 hours if you want to practise the timing. Use the self-check cells. Ask your lecturer about anything unclear before the sitting. A solved walk-through is not given: the point is that you can do it yourself.

This project uses only what you learned in Weeks 1 to 11. You work on `engineering_data.csv` (motor winding temperature, in **deg C**), and answer two questions: **Question 1** how large will the motor winding temperature be on equipment the model has never met, and **Question 2** which rows should raise a warning. Group E6 is the test set and is opened **once**.

## How to work
1. Fill in the identity cell below (in the issued notebook it is filled in for you; check it) and run it. Then run the supplied cell **Prepare the data and the libraries**. It creates your CSV, so you need no upload.
2. Work from top to bottom. Replace each `TODO` and each `None`. Keep the function names, the argument order and the variable names exactly as printed. Run each cell and read what it prints.
3. Run the **self-check** cells to see which functions still need work. They are not marked.
4. Part B is a form (the `MEMO` cell). Nothing is written as an essay.
5. Choose **Runtime, then Restart session and run all**. Fix any error, then save with **File, then Save**.
6. Run the last cell, click **Submit test**, open the form link, check your details and press **Submit** there. Submission is finished only when the form says your response has been recorded.

## Time plan (180 minutes)
| Step | Time |
|---|---|
| Setup: sign in, open your notebook, run the supplied cells | 10 min |
| Tasks 1 to 3: cleaning, the split, the three models | 45 min |
| Tasks 4 to 6: warnings, cross-validation, ablation | 45 min |
| Tasks 7 and 8: choose a model, open the test set once | 15 min |
| Task 9: the Part B memo form | 25 min |
| Check, restart and run all, submit | 25 min |
| Spare time (breaks, reconnecting) | 15 min |

## How it is marked
Everything is marked by the lecturer's computer. No person reads your notebook. Your functions are run on **other data that you cannot see** (hidden checks) and your results on your own data file are compared with the correct values. Of the 30 Part A marks, 24 are for the functions and your results, and 6 are for how you worked: the test set opened once and only after you chose, no data leakage (fit preprocessing on the training set only), the same answer every time (keep `random_state=17`), and readable charts. Part B gives 15 marks for the memo form. A list of notebooks that look unusual is checked by the lecturer afterwards.

**Rules (the same rules as Practical Test 1).** Individual work. **Permitted resources:** your lecture slides, your lab notebooks from Classes 1 to 4, the revision guide, the cheatsheet and your own course notes. Write the code yourself.
**No AI or outside help:** do not use Gemini, ChatGPT, Copilot or any other AI tool to generate, complete, rewrite, explain or debug your answers. Do not get answers from another person, collaborate or copy someone else's work.
**Understanding check:** you may be asked to explain your submitted code, or make a small change to it, without AI or another person's help.
**Declaration:** by submitting, I confirm that I completed this project myself without AI assistance or another person's help, using only the permitted resources. I can explain and change the code I submitted.


In [ ]:
FULL_NAME = ""
EMAIL = ""
STUDENT_ID = ""   # your Student Registration Number exactly as on your university record
COHORT = "CML3111"  # Do not edit: identifies your course for marking.
TEST_ID = "ML_PROJECT_REHEARSAL"  # Do not edit: identifies this assessment.
VARIANT = "P"  # Do not edit.


## Your data
Each row is one synthetic (invented for teaching) observation of **motor winding temperature**.

| Column | Meaning |
|---|---|
| record_id | Row ID. A repeated ID means a duplicate row. record_id is never a time. |
| equipment_id | Which equipment: E1 to E6. Repeats across rows. E6 is the test set. |
| load_pct | Load in percent (a feature). Valid range 0 to 100, both included. |
| ambient_c | Ambient temperature in degrees Celsius (a feature). May be a missing value. |
| target | The target variable: motor winding temperature, in **deg C**. May be a missing value. |

A **record** is one row as a dictionary. A **list of records** is several rows. Your functions take and return these. Missing values are `None`. Calculate from the lists you are given. Do not use the global tables inside a function, because the marked checks give your functions other data.

In [ ]:
#@title Prepare the data and the libraries - Do not edit this cell
# IUEA-SETUP-CELL  (supplied: do not edit)
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score

TARGET_NAME = 'motor winding temperature'
TARGET_UNIT = 'deg C'
LIMIT = 80.0          # a row is a WARNING (1) when its target is at or above this limit, in deg C
MAX_MAE = 2.0          # the client accepts a typical prediction error (MAE) of at most this many deg C (invented teaching number)
MAX_MS = 50                 # and a prediction time of at most this many milliseconds (invented teaching number)
LATENCY_MS = {'baseline': 1, 'linear': 3, 'tree': 30}   # invented teaching numbers
MISSED_COST, FALSE_ALARM_COST = 5, 1   # invented teaching costs (a false negative costs 5, a false positive costs 1)

_CSV_TEXT = """record_id,equipment_id,load_pct,ambient_c,target
1,E1,52.24,26.92,66.619
2,E2,35.65,30.3,59.39
3,E3,48.42,29.46,66.336
4,E4,90.75,18.96,
5,E5,87.55,19.26,83.081
6,E6,64.9,29.17,76.83
7,E1,93.09,25.05,92.94
8,E2,39.88,28.23,58.853
9,E3,-5,32.0,81.307
10,E4,44.76,24.81,58.053
11,E5,36.79,32.98,62.665
12,E6,59.48,18.14,60.995
13,E1,82.76,26.8,87.828
14,E2,61.03,,64.835
15,E3,57.81,21.31,64.449
16,E4,63.29,32.42,80.108
17,E5,32.26,24.56,49.525
18,E6,88.06,26.66,90.527
19,E1,70.08,18.61,69.042
20,E2,30.94,26.44,50.645
21,E3,80.99,20.08,78.258
22,E4,64.35,21.84,68.418
23,E5,64.15,18.87,65.799
24,E6,69.57,28.7,80.822
25,E1,80.06,23.01,81.788
26,E2,50.17,32.69,70.998
27,E3,40.7,31.77,63.428
28,E4,62.61,32.49,79.883
29,E5,70.94,27.21,80.168
30,E6,92.69,31.9,99.654
31,E1,40.82,19.21,50.914
32,E2,56.08,27.67,68.946
33,E3,47.73,31.62,67.515
34,E4,54.12,29.78,70.858
35,E5,67.71,19.53,68.829
36,E6,28.97,31.08,55.371
37,E1,42.28,19.21,49.547
38,E2,63.62,19.56,67.265
39,E3,57.29,27.4,71.022
40,E4,58.53,22.38,65.797
41,E5,53.52,33.87,74.085
42,E6,66.93,25.49,74.283
43,E1,56.22,32.45,75.239
44,E2,83.88,18.65,78.793
45,E3,67.41,34.72,85.039
46,E4,74.71,23.45,77.375
47,E5,53.43,30.55,70.445
48,E6,73.39,30.7,85.623
49,E1,37.72,26.57,55.479
50,E2,23.24,20.43,40.074
51,E3,65.61,18.98,66.355
52,E4,50.15,34.1,73.849
53,E5,45.62,31.03,64.723
54,E6,55.68,34.08,75.969
55,E1,24.55,25.32,45.726
56,E2,35.13,18.83,45.879
57,E3,88.88,24.11,88.654
58,E4,91.63,21.15,87.904
59,E5,25.91,27.98,50.247
60,E6,56.07,19.17,60.306
61,E1,32.35,32.03,57.773
62,E2,77.86,31.86,89.519
63,E3,69.22,26.57,77.689
64,E4,38.15,22.71,52.622
65,E5,43.01,20.71,54.19
66,E6,58.19,34.43,77.291
67,E1,21.69,19.61,36.895
68,E2,87.65,26.14,89.994
69,E3,88.16,20.84,84.069
70,E4,70.33,34.3,86.977
71,E5,76.78,34.61,91.095
72,E6,42.88,25.99,58.391
73,E1,82.46,25.93,86.695
74,E2,64.48,32.5,81.016
75,E3,69.77,21.98,71.937
76,E4,86.58,30.28,94.562
77,E5,75.5,30.14,85.328
78,E6,23.77,18.81,37.812
79,E1,32.58,28.9,55.647
80,E2,80.65,32.79,93.324
81,E3,91.31,29.21,97.849
82,E4,41.9,29.47,61.35
83,E5,63.17,18.5,65.028
84,E6,90.41,22.44,86.6
85,E1,35.82,23.77,51.047
86,E2,87.55,24.51,88.612
87,E3,86.74,26.08,89.462
88,E4,63.2,27.27,74.548
89,E5,36.57,25.97,53.129
90,E6,34.32,32.57,59.222
1,E1,52.24,26.92,66.619
"""
with open("engineering_data.csv", "w") as _f:          # your own copy of the data file: the same rows every time you run this cell
    _f.write(_CSV_TEXT)

def records_from_table(table):
    """A list of dictionaries, one per row. Missing values (NaN) become None."""
    objects = table.astype(object)
    return objects.where(table.notna(), None).to_dict("records")

raw_table = pd.read_csv("engineering_data.csv")
all_records = records_from_table(raw_table)
development_records = [r for r in all_records if r["equipment_id"] != "E6"]   # E6 is the TEST SET: nothing you write may use it before the very end
print("Rows in the file:", len(raw_table))
print("Rows you may develop with (groups E1 to E5):", len(development_records))
print(raw_table.head())


## Task 1: Clean the rows and look at them (2 automatic marks, Weeks 3 to 7)

A **record** is one row written as a dictionary: `{"record_id": 1, "equipment_id": "E1", "load_pct": 59.15, "ambient_c": 31.71, "target": 52.5}`. A missing value (a blank cell, `NaN` in pandas) is `None`.

`clean_rows(records)` returns a cleaned **copy** of the records, in the same order:
- keep only the first row for each `record_id`;
- drop a row whose `target` is `None`, or whose `load_pct` is `None` or outside 0 to 100 (0 and 100 are valid);
- **do not fill** `ambient_c`. A missing value stays `None`. The models do the imputation later, using the training set only, so that nothing leaks (data leakage).

Never change the list you were given. Then use your function on the development rows and draw **Chart 1**: `load_pct` across, `target` up, with a title and both axis names, and the unit (deg C).

In [ ]:
def clean_rows(records):
    """Return a cleaned copy of the records (see the task text above)."""
    # TODO 1: make an empty list called kept and an empty set called seen.
    # TODO 2: loop over records. If record["record_id"] is already in seen, skip the row (continue). Otherwise add the id to seen.
    # TODO 3: skip the row if target is None, or load_pct is None, or load_pct is below 0 or above 100.
    # TODO 4: add dict(record) (a copy) to kept. Do not touch ambient_c.
    return None   # TODO 5: return kept instead


In [ ]:
development_rows = None      # TODO: call clean_rows on development_records
print("Rows kept:", None)    # TODO: replace None with the number of rows in development_rows

figure, axes = plt.subplots()
# TODO: axes.scatter(...): load_pct on the x axis, target on the y axis (hint: table = pd.DataFrame(development_rows))
# TODO: axes.set_title(...), axes.set_xlabel(...) and axes.set_ylabel(...). Write the unit in the y label.
chart_1 = figure
plt.show()


## Task 2: Split by whole equipment groups (2 automatic marks, Week 10)

`group_split(records)` returns a dictionary with three lists of `record_id` values, each in the order the records came in:
- `"train"`: groups E1, E2, E3 and E4 (the **training set**);
- `"validation"`: group E5 (the **validation set**);
- `"test"`: group E6 (the **test set**).

Never use the target to decide where a row goes. Whole machines go together, because a new machine must be a machine the model has never met. Your development rows have no E6 rows, so their `"test"` list is empty. That is correct.

In [ ]:
def group_split(records):
    """Return {"train": [...], "validation": [...], "test": [...]} (lists of record_id, in input order)."""
    # TODO: for each record, look at record["equipment_id"] and put record["record_id"] in the right list.
    return None


In [ ]:
split_ids = None     # TODO: group_split(development_rows)
train_rows = None    # TODO: the development rows whose record_id is in split_ids["train"]
valid_rows = None    # TODO: the development rows whose record_id is in split_ids["validation"]
print("Training rows:", None, "| validation rows:", None)   # TODO: replace the Nones with the two counts


## Task 3: Question 1: a baseline model, a linear regression and a decision tree (5 automatic marks, Weeks 8 and 9)

**Question 1.** How many deg C of motor winding temperature will a motor show at a given load and ambient temperature, for equipment the model has never met?

`fit_predict_regression(training_records, feature_records)` trains (fits) three models on the training records and predicts for the feature records. Use only `load_pct` then `ambient_c`, in that order. The feature records have **no target**.
- every model is a pipeline that first does **imputation** of missing values with the **median of the training rows** (`SimpleImputer(strategy="median")`), then fits;
- `"baseline"`: the baseline model, which always predicts the mean training target (`DummyRegressor`);
- `"linear"`: multiple linear regression (two features), `LinearRegression`;
- `"tree"`: a decision tree, `DecisionTreeRegressor(max_depth=4, random_state=17)`. The `max_depth` is a **hyperparameter**: a setting you choose before training.

Return a dictionary with three lists of predictions, in the order of the feature records. The baseline model must use the **training** rows only: a baseline built with validation or test targets has **leaked**.

`regression_errors(actual, predicted)` returns `{"mae": ..., "rmse": ...}` over the same pairs, not rounded. A **residual** is actual minus predicted. MAE (mean absolute error) is the mean of the absolute residuals. RMSE (root mean squared error) is the square root of the mean of the squared residuals.

In [ ]:
def fit_predict_regression(training_records, feature_records):
    """Return {"baseline": [...], "linear": [...], "tree": [...]}."""
    # TODO 1: table = pd.DataFrame(training_records); new = pd.DataFrame(feature_records)
    # TODO 2: X_train = the two feature columns of table; y_train = table["target"]; X_new = the same two columns of new.
    # TODO 3: a dictionary of three pipelines. Each is make_pipeline(SimpleImputer(strategy="median"), <the model>).
    # TODO 4: loop over the dictionary: fit on (X_train, y_train), predict X_new, and store predictions.tolist() under the model's name.
    return None


In [ ]:
def regression_errors(actual, predicted):
    """Return {"mae": ..., "rmse": ...}, not rounded."""
    # TODO: use numpy: residuals = np.array(actual) - np.array(predicted); MAE uses np.abs and the mean; RMSE uses squares, the mean, then np.sqrt.
    return None


In [ ]:
# Evaluate the three models on the validation set (group E5). Keep the names: validation_predictions, validation_mae and validation_rmse.
validation_features = None    # TODO: the validation rows without the "target" key
validation_actual = None      # TODO: the list of validation targets
validation_predictions = None # TODO: fit_predict_regression(train_rows, validation_features)
validation_mae = None         # TODO: a dictionary {"baseline": ..., "linear": ..., "tree": ...} of MAE values, from regression_errors
validation_rmse = None        # TODO: the same with RMSE
print("Validation MAE  (" + TARGET_UNIT + "):", validation_mae)
print("Validation RMSE (" + TARGET_UNIT + "):", validation_rmse)

figure, axes = plt.subplots()
# TODO: a bar chart of the three MAE values, with a title, a y label that includes the unit, and the model names on the x axis.
chart_2 = figure
plt.show()


## Task 4: Question 2: which rows should raise a warning? (5 automatic marks, Week 9)

**Question 2.** A row is a **warning** (1, the positive class) when its target is at or above `LIMIT` (80 deg C) and **no warning** (0, the negative class) below it. Which rows should raise a warning?

`warning_probabilities(training_records, feature_records, limit)` returns the probability of a warning, for each feature record, from a pipeline of `SimpleImputer(strategy="median")`, `StandardScaler()` and `LogisticRegression(max_iter=1000)` (binary classification), fitted on the training records only. The label is `1` when the training target is `>= limit`. Use `load_pct` then `ambient_c`.

`classification_scores(actual, probability, threshold)` predicts a warning when the probability is `>= threshold` (the **decision threshold**), then returns a dictionary with the confusion matrix counts `tp`, `fp`, `fn`, `tn` (true positives, false positives, false negatives, true negatives) and `precision`, `recall`, `f1`, `cost` and `auc`. A false negative (a missed warning) costs `MISSED_COST` and a false positive (a false alarm) costs `FALSE_ALARM_COST` (invented teaching costs), so `cost = 5 * fn + 1 * fp`. When a ratio has a zero denominator, use 0. `auc` is the ROC AUC of the probabilities and does not depend on the threshold.

Then try decision thresholds 0.3, 0.5 and 0.7 on the validation set, and keep the one with the lowest cost as `chosen_threshold` (if two tie, take the higher one).

In [ ]:
def warning_probabilities(training_records, feature_records, limit):
    """Return a list of warning probabilities, one per feature record."""
    # TODO 1: tables as before; y_train = (training target >= limit) turned into 0 and 1 (astype(int)).
    # TODO 2: model = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(max_iter=1000)); fit it.
    # TODO 3: predict_proba(X_new) gives two columns: column 1 is the probability of a warning. Return it as a list of floats.
    return None


In [ ]:
def classification_scores(actual, probability, threshold):
    """Return tp, fp, fn, tn, precision, recall, f1, cost and auc."""
    # TODO 1: predicted = 1 where the probability is at or above the threshold, else 0 (a list comprehension).
    # TODO 2: tn, fp, fn, tp = confusion_matrix(actual, predicted).ravel()   (see Class 2)
    # TODO 3: precision_score, recall_score and f1_score with zero_division=0; roc_auc_score(actual, probability).
    # TODO 4: cost = MISSED_COST * fn + FALSE_ALARM_COST * fp. Turn numpy numbers into int() and float() before returning.
    return None


In [ ]:
warning_actual = None        # TODO: 1 or 0 for each validation row: is the target at or above LIMIT?
warning_probability = None   # TODO: warning_probabilities(train_rows, validation_features, LIMIT)
threshold_table = {}         # TODO: for each threshold in 0.3, 0.5, 0.7: threshold_table[threshold] = classification_scores(...)
warning_auc = None           # TODO: the "auc" value of any one entry of threshold_table
chosen_threshold = None      # TODO: the threshold with the lowest cost (if two tie, the higher threshold)
for threshold, scores in threshold_table.items():
    print(threshold, scores)
print("Chosen threshold:", chosen_threshold)


## Task 5: Can you trust it? Group cross-validation (3 automatic marks, Week 10)

One validation set of about 15 rows can mislead. **Group cross-validation** (K-fold cross-validation where each fold is a whole equipment group, the idea of `GroupKFold`) gives each machine a turn as the hidden machine.

`group_cv_mae(records)` receives clean records with **no missing values** and uses whole equipment groups. Hold out each of E1, E2, E3, E4 and E5 in turn (E6 is never used). For each fold, fit `LinearRegression` on `load_pct` and `ambient_c` using the other four groups, and compute the MAE on the held-out group. Return `{"folds": [...], "mean": ..., "spread": ...}`: the five MAEs in the order E1 to E5, their mean, and their sample standard deviation (`ddof=1`). Do not round.

**Data leakage check, in your own head:** the imputer, the StandardScaler (standardisation) and the baseline model's mean are all fitted on the training rows only. If any of them had seen validation or test rows, the scores would look better than they are.

In [ ]:
def group_cv_mae(records):
    """Return {"folds": [five MAEs, E1 to E5], "mean": ..., "spread": ...}."""
    # TODO 1: table = pd.DataFrame(records)
    # TODO 2: for group in ["E1", "E2", "E3", "E4", "E5"]: train = rows of the OTHER groups in E1 to E5, test = the rows of group.
    # TODO 3: fit LinearRegression on the two feature columns of train; the MAE on test is np.mean(np.abs(...)).
    # TODO 4: mean = np.mean(folds); spread = np.std(folds, ddof=1).
    return None


In [ ]:
complete_rows = None    # TODO: the development rows whose ambient_c is not None
cv_summary = None       # TODO: group_cv_mae(complete_rows)
print("Cross-validated MAE (" + TARGET_UNIT + "):", cv_summary)


## Task 6: Does a new column help? An ablation study (2 automatic marks, Week 11)

`ablation_mae(training_records, validation_records)` is a small **ablation study**: it compares two linear regression models on the same rows. Both records lists are complete (no missing values). Fit `LinearRegression` on the training records twice: with the two base features (`load_pct`, `ambient_c`), and with a third column, the **interaction feature** `load_pct * ambient_c`. Score both on the validation records. Return `{"base_mae": ..., "extended_mae": ..., "improvement": base_mae - extended_mae}`, not rounded. A positive improvement means the new feature helped.

In [ ]:
def ablation_mae(training_records, validation_records):
    """Return base_mae, extended_mae and improvement."""
    # TODO 1: two tables. Build the extra column on both: table["load_x_ambient"] = table["load_pct"] * table["ambient_c"].
    # TODO 2: fit LinearRegression with the base columns, then with the base columns plus load_x_ambient. Same rows, same measure.
    return None


In [ ]:
complete_train = None    # TODO: the rows of train_rows whose ambient_c is not None
complete_valid = None    # TODO: the rows of valid_rows whose ambient_c is not None
ablation_result = None   # TODO: ablation_mae(complete_train, complete_valid)
print("Ablation (" + TARGET_UNIT + "):", ablation_result)


## Task 7: Choose one model by a rule (1 automatic marks, Week 11)

`choose_model(candidates, max_mae, max_ms)` takes a list of dictionaries with `name`, `validation_mae` and `latency_ms`. Keep the candidates whose `validation_mae <= max_mae` **and** `latency_ms <= max_ms` (both limits are inclusive). Among those, choose the smallest `validation_mae`; break a tie by the smaller `latency_ms`, then by the name in alphabetical order. Return the chosen name, or `None` if nobody qualifies. Never use the test set to choose.

The client for this project is the maintenance manager who pays for the motors. The client accepts a typical prediction error of at most `MAX_MAE` (2 deg C) and a prediction time of at most `MAX_MS` (50 ms). Both numbers are invented for teaching.

In [ ]:
def choose_model(candidates, max_mae, max_ms):
    """Return the chosen model name, or None."""
    # TODO: filter with a list comprehension, then sort with key=lambda c: (c["validation_mae"], c["latency_ms"], c["name"]).
    return None


In [ ]:
candidate_table = []     # TODO: one dictionary per model (baseline, linear, tree): name, validation_mae (from your Task 3 results), latency_ms (from LATENCY_MS)
chosen_model = None      # TODO: choose_model(candidate_table, MAX_MAE, MAX_MS)
print(candidate_table)
print("Chosen model:", chosen_model)


## Task 8: Open the test set, once (4 automatic marks, Week 11)

Group E6 is the **test set**: a different piece of equipment that nothing so far has used. It gives **one** honest estimate for the model you chose.

How it works. The supplied cell below is marked Do not edit this cell. Run it once to define `open_test_set(model_name, threshold)`. Then call it **one time**, with the model you chose in Task 7 and the decision threshold you chose in Task 4. It trains your functions on all the development rows (E1 to E5), evaluates on E6, prints the result and writes your choice to a small file. If you call it again, the notebook counts a second opening: you lose the "opened once" mark even when the choice is the same, and you lose more if the choice is different.

You must not use E6 to choose a model, a threshold or anything else, and you must not look at it before this call. Report whatever the result is in your Part B form.

In [ ]:
#@title Open the test set (run it ONCE, at the very end) - Do not edit this cell
# IUEA-TEST-CELL  (supplied: do not edit)
import json as _json
TEST_OPEN_LOG = []
_TEST_FILE = "test_first_opening.json"

def open_test_set(model_name, threshold):
    """Evaluate ONE chosen model and ONE chosen decision threshold on the test set (group E6), with the functions you wrote. Returns a dictionary."""
    if model_name not in ("baseline", "linear", "tree"):
        raise ValueError('model_name must be "baseline", "linear" or "tree"')
    if not isinstance(threshold, (int, float)) or not 0 < threshold < 1:
        raise ValueError("threshold must be a number between 0 and 1")
    first = _json.load(open(_TEST_FILE)) if os.path.exists(_TEST_FILE) else None
    if first is None:
        _json.dump({"model": model_name, "threshold": threshold}, open(_TEST_FILE, "w"))
    TEST_OPEN_LOG.append([model_name, threshold])
    changed = first is not None and (first["model"] != model_name or first["threshold"] != threshold)
    if changed:
        print("WARNING: THE TEST SET WAS ALREADY OPENED with model", first["model"], "and threshold", first["threshold"],
              "- opening it again with a different choice means it is no longer an honest test.")
    table = pd.read_csv("engineering_data.csv").drop_duplicates("record_id")
    table = table[table["target"].notna() & table["load_pct"].between(0, 100)]
    development = records_from_table(table[table["equipment_id"] != "E6"])
    held_out = records_from_table(table[table["equipment_id"] == "E6"])
    features = [{k: v for k, v in r.items() if k != "target"} for r in held_out]
    actual = [r["target"] for r in held_out]
    errors = regression_errors(actual, fit_predict_regression(development, features)[model_name])
    probability = warning_probabilities(development, features, LIMIT)
    warning = classification_scores([1 if v >= LIMIT else 0 for v in actual], probability, threshold)
    outcome = {"model": model_name, "threshold": threshold, "n_rows": len(held_out), "mae": errors["mae"], "rmse": errors["rmse"],
               "warning": warning, "changed_after_opening": bool(changed)}
    print("Test set opened with", model_name, "and threshold", threshold, "on", len(held_out), "rows")
    print("  MAE", round(errors["mae"], 3), TARGET_UNIT, "| RMSE", round(errors["rmse"], 3), TARGET_UNIT)
    print("  warnings: precision", round(warning["precision"], 3), "recall", round(warning["recall"], 3), "F1", round(warning["f1"], 3), "AUC", round(warning["auc"], 3))
    return outcome


In [ ]:
test_result = None     # TODO: open_test_set(chosen_model, chosen_threshold)


## Self-check (not marked)
Run this supplied cell to see which functions still need work. It uses tiny examples with known answers. The marked checks use other data.

In [ ]:
# This cell is a SELF-CHECK. It is not marked. It uses tiny examples with known answers, so you can see at once if a function is wrong.
# The marked (hidden) checks run your functions on OTHER data you cannot see, so passing here does not mean full marks.
def _try(label, test):
    try:
        ok = bool(test())
    except Exception as error:
        ok = False
    print(("PASS  " if ok else "TODO  ") + label)
    return ok

_small = [dict(record_id=1, equipment_id="E1", load_pct=50.0, ambient_c=20.0, target=10.0),
          dict(record_id=1, equipment_id="E1", load_pct=50.0, ambient_c=20.0, target=10.0),
          dict(record_id=2, equipment_id="E2", load_pct=120.0, ambient_c=22.0, target=12.0),
          dict(record_id=3, equipment_id="E5", load_pct=40.0, ambient_c=None, target=None),
          dict(record_id=4, equipment_id="E6", load_pct=100.0, ambient_c=None, target=20.0)]
_checks = []
_checks.append(_try("clean_rows keeps ids 1 and 4 only, in order, and leaves the missing ambient_c as None",
                    lambda: [r["record_id"] for r in clean_rows(_small)] == [1, 4] and clean_rows(_small)[1]["ambient_c"] is None))
_checks.append(_try("clean_rows does not change the list you give it", lambda: (len(_small) == 5 and clean_rows(_small) is not _small)))
_checks.append(_try("group_split puts E1 in train, E5 in validation, E6 in test",
                    lambda: group_split(_small) == {"train": [1, 1, 2], "validation": [3], "test": [4]}))
_toy = [dict(record_id=i, equipment_id="E1", load_pct=10.0 * i, ambient_c=20.0, target=2.0 * i) for i in range(1, 7)]
_new = [dict(record_id=9, equipment_id="E2", load_pct=35.0, ambient_c=None)]
_checks.append(_try("fit_predict_regression returns baseline, linear and tree, one prediction per feature record",
                    lambda: sorted(fit_predict_regression(_toy, _new)) == ["baseline", "linear", "tree"] and all(len(v) == 1 for v in fit_predict_regression(_toy, _new).values())))
_checks.append(_try("the baseline model predicts the mean of the TRAINING targets (7.0)", lambda: abs(fit_predict_regression(_toy, _new)["baseline"][0] - 7.0) < 1e-9))
_checks.append(_try("regression_errors([1, 3], [2, 5]) has mae 1.5 and rmse sqrt(2.5)",
                    lambda: abs(regression_errors([1, 3], [2, 5])["mae"] - 1.5) < 1e-9 and abs(regression_errors([1, 3], [2, 5])["rmse"] - 2.5 ** 0.5) < 1e-9))
_checks.append(_try("classification_scores on a known case (tp 1, fp 1, fn 1, tn 1, cost 6, auc 0.75)",
                    lambda: (lambda s: (s["tp"], s["fp"], s["fn"], s["tn"], s["cost"], round(s["auc"], 2)) == (1, 1, 1, 1, 6, 0.75))(classification_scores([1, 0, 1, 0], [0.9, 0.8, 0.2, 0.1], 0.5))))
_checks.append(_try("warning_probabilities gives numbers between 0 and 1",
                    lambda: all(0 <= p <= 1 for p in warning_probabilities(_toy, _new, 8.0))))
_checks.append(_try("choose_model picks the smallest error that meets both limits",
                    lambda: choose_model([dict(name="a", validation_mae=2, latency_ms=3), dict(name="b", validation_mae=1, latency_ms=30)], 2, 10) == "a"))
_checks.append(_try("group_cv_mae returns five folds, a mean and a spread",
                    lambda: len(group_cv_mae(complete_rows)["folds"]) == 5 and {"mean", "spread"} <= set(group_cv_mae(complete_rows))))
_checks.append(_try("ablation_mae returns base_mae, extended_mae and improvement",
                    lambda: {"base_mae", "extended_mae", "improvement"} <= set(ablation_mae(complete_train, complete_valid))))
for _name in ["validation_mae", "cv_summary", "warning_auc", "ablation_result", "candidate_table", "chosen_model", "chosen_threshold"]:
    _checks.append(_try("the variable " + _name + " has been calculated (it is not None)", lambda: globals().get(_name) is not None))
_checks.append(_try("the test set has been opened exactly once", lambda: len(TEST_OPEN_LOG) == 1))
_checks.append(_try("Chart 1 and Chart 2 each have a title, named axes and the unit",
                    lambda: all(f.axes[0].get_title() and f.axes[0].get_xlabel() and f.axes[0].get_ylabel() for f in (chart_1, chart_2))))
print()
print(sum(_checks), "of", len(_checks), "self-checks pass. Now choose Runtime, Restart session and run all, and check that nothing stops with an error.")


## Task 9: Part B, the decision memo form (15 automatic marks)
Your memo is a **form**, not an essay. You write it for **the maintenance manager who pays for the motors**. Fill in the dictionary `MEMO` in the next cell. Every value is checked by the marking computer; nothing is read by a person. Use your own numbers, with at least two decimals, in **deg C**.

| Field | What to enter | Marks |
|---|---|---|
| `recommended_model`, `recommended_threshold` | One of the models you trained (`"baseline"`, `"linear"`, `"tree"`) and one decision threshold (0.3, 0.5 or 0.7). They must be the model and threshold your own rule and cost table chose. | 3 |
| `validation_mae` | The validation MAE of each of the three models, as your notebook computed them. | 3 |
| `test_mae`, `test_recall` | The MAE and the warning recall on the test set from your ONE opening (recall is between 0 and 1). The marking computer recomputes them from your own functions. | 2 |
| `cv_mean_mae` | The mean MAE from your group cross-validation. | 1 |
| `limits` | The letters of the **three** statements below that are **true limits of this study**. Three different letters. One mark for each true one. | 3 |
| `next_step` | One letter: the best next step. | 1 |
| `answer_choice`, `answer_test_set` | Two short written answers of 8 to 80 words each. One mark each. | 2 |

**Limits.** Choose the three statements that are TRUE and are limits of this study (some statements are false):
- **A.** The test MAE proves that the model is safe on every motor the client owns.
- **B.** The baseline model should be left out of the comparison because it uses no features.
- **C.** Because the test set was opened only once, the test metrics have no uncertainty at all.
- **D.** RMSE is always smaller than MAE, so MAE is the more severe measure and should be the only one reported.
- **E.** The client limits, the decision threshold options and the costs (5 and 1) are invented teaching numbers; a real client would set their own.
- **F.** The test set is a single equipment group, so the test metrics are one estimate that may be luckier or unluckier than the truth.
- **G.** The model with the lowest training error is always the best model for new equipment.
- **H.** The validation set is one equipment group with only about 15 rows, so the validation errors could change a lot with a different group.
- **I.** Fitting the imputer on the training set only makes the validation results unreliable.
- **J.** The data are synthetic (invented for teaching), so the numbers say nothing certain about real motors.

**Next step.** Choose the one best next step:
- **A.** Move the test set into the training set so the model learns from more rows, and report the training error.
- **B.** Tune the maximum depth of the decision tree until its test MAE is lowest, then report that test MAE.
- **C.** Deploy the chosen model now, because the test MAE is below the client limit.
- **D.** Remove the baseline model from the report so the results look better.
- **E.** Collect real measurements from more motors and validate the chosen model again with group cross-validation before any decision.

**Written answers.** Write in your own words, in full sentences.
- `answer_choice`: In one or two sentences: why do you recommend this model? Use a number from the validation set and say what you compared it with.
- `answer_test_set`: In one or two sentences: why is the test set opened only once, at the very end, and never used to choose the model?

The lettering of these lists is different for each student, so a list of letters from a classmate will not match your lists.

In [ ]:
#@title Part B: the decision memo form (fill in every field; do not rename a field)
MEMO = {
    "recommended_model": "",                  # "baseline", "linear" or "tree"
    "recommended_threshold": None,            # 0.3, 0.5 or 0.7
    "validation_mae": {"baseline": None, "linear": None, "tree": None},   # your validation MAE values, in deg C
    "cv_mean_mae": None,                      # your mean cross-validated MAE, in deg C
    "test_mae": None,                         # the test MAE from your ONE opening, in deg C
    "test_recall": None,                      # the test recall from your ONE opening (0 to 1)
    "limits": ["", "", ""],                   # three different letters from the Limits list
    "next_step": "",                          # one letter from the Next step list
    "answer_choice": "",                      # one or two sentences
    "answer_test_set": "",                    # one or two sentences
}


## Memo self-check (not marked)
Run this supplied cell after you fill in the form. It tells you which fields are empty and whether your numbers agree with your own notebook.

In [ ]:
# MEMO SELF-CHECK (not marked): did you fill in every field, and do your numbers agree with your own notebook?
# It does not tell you whether your choices are the best ones. The marking computer checks that.
def _memo_try(label, test):
    try:
        ok = bool(test())
    except Exception:
        ok = False
    print(("PASS  " if ok else "TODO  ") + label)
    return ok

def _near(a, b):
    return isinstance(a, (int, float)) and isinstance(b, (int, float)) and abs(a - b) <= 0.011

_memo_checks = [
    _memo_try("recommended_model is one of the models you trained", lambda: MEMO["recommended_model"] in validation_mae),
    _memo_try("recommended_threshold is 0.3, 0.5 or 0.7", lambda: MEMO["recommended_threshold"] in (0.3, 0.5, 0.7)),
    _memo_try("validation_mae has your three validation MAE values (two decimals are enough)", lambda: all(_near(MEMO["validation_mae"][k], validation_mae[k]) for k in ("baseline", "linear", "tree"))),
    _memo_try("cv_mean_mae is your cross-validated mean MAE", lambda: _near(MEMO["cv_mean_mae"], cv_summary["mean"])),
    _memo_try("test_mae is the test MAE you saw when you opened the test set", lambda: _near(MEMO["test_mae"], test_result["mae"])),
    _memo_try("test_recall is the test recall you saw (between 0 and 1)", lambda: _near(MEMO["test_recall"], test_result["warning"]["recall"])),
    _memo_try("limits has three different letters", lambda: len({str(x).strip().upper() for x in MEMO["limits"] if str(x).strip()}) == 3),
    _memo_try("next_step is one letter", lambda: len(str(MEMO["next_step"]).strip()) == 1),
    _memo_try("answer_choice has 8 to 80 words", lambda: 8 <= len(str(MEMO["answer_choice"]).split()) <= 80),
    _memo_try("answer_test_set has 8 to 80 words", lambda: 8 <= len(str(MEMO["answer_test_set"]).split()) <= 80),
]
print()
print(sum(_memo_checks), "of", len(_memo_checks), "memo self-checks pass.")


## End of the rehearsal
Nothing is submitted from this notebook. On the day, the real notebook has a last cell that saves a snapshot and opens the submission form, and a countdown at the top.

Before the sitting, check that you can: finish every function and see PASS in the self-check; run **Runtime, Restart session and run all** with no error; fill in every `MEMO` field and see 10 of 10 memo self-checks pass; and explain, in your own words, what each function does and why the test set is opened only once.